In [48]:
# %% [Cell 1: Setup & Imports]
import os
import time
import pandas as pd
import numpy as np
import yfinance as yf
import torch
import torch.nn.functional as F
from newsapi import NewsApiClient
from datetime import datetime, timedelta
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import tensorflow as tf
from tensorflow.keras import layers, models

# Suppress Warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
pd.options.mode.chained_assignment = None

In [49]:
# %% [Cell 2: Initialize FinBERT]
# This loads the model into memory once.
print("Loading FinBERT Model...")
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name)

def get_sentiment(text):
    """Processes text and returns a score between -1 and 1."""
    if not text or pd.isna(text): return 0
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = finbert_model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1).numpy()[0]
    # FinBERT labels: [Positive, Negative, Neutral]
    return probs[0] - probs[1]

Loading FinBERT Model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6952.97it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
# %% [Cell 3: Data Acquisition & Feature Engineering]
# SET YOUR KEY HERE
NEWS_API_KEY = 'b13947d321a24e089c0af84aedfc47a9'
newsapi = NewsApiClient(api_key=NEWS_API_KEY)
ticker_symbol = "AMZN"

# Define Date Range (Free tier allows ~27 days)
end_date = datetime.now()
start_date = end_date - timedelta(days=27)
from_str = start_date.strftime('%Y-%m-%d')

print(f"Fetching news and prices starting from {from_str}...")

# 1. Fetch Paginated News (5 pages = 500 articles)
processed_news = []
for page in range(1, 6):
    try:
        batch = newsapi.get_everything(
            q='Amazon AND (stock OR shares OR earnings)',
            from_param=from_str,
            language='en',
            sort_by='publishedAt',
            page=page,
            page_size=100
        )
        articles = batch.get('articles', [])
        if not articles: break

        for art in articles:
            processed_news.append({
                'Date': art['publishedAt'][:10],
                'Sentiment': get_sentiment(art['title'])
            })
        print(f"Processed News Page {page}...")
        time.sleep(0.1)
    except Exception as e:
        print(f"API Limit reached or error: {e}")
        break

# 2. Aggregate Sentiment
sentiment_df = pd.DataFrame(processed_news).groupby('Date').agg(
    Sentiment_Score=('Sentiment', 'mean'),
    Article_Count=('Sentiment', 'count')
).reset_index()
sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])

# 3. Fetch Prices
prices = yf.download(ticker_symbol, start=from_str)
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)
prices = prices[['Close']].reset_index()
prices['Return'] = prices['Close'].pct_change()
prices['Date'] = pd.to_datetime(prices['Date'])

# 4. Merge & Bridge Weekends (Forward-fill sentiment)
merged = pd.merge(prices, sentiment_df, on='Date', how='left')
merged['Sentiment_Score'] = merged['Sentiment_Score'].fillna(0)
merged['Article_Count'] = merged['Article_Count'].fillna(0)

# 5. Create Lags (Yesterday's News predicts Today's Price)
merged['Prev_Sentiment'] = merged['Sentiment_Score'].shift(1)
merged['Prev_Count'] = merged['Article_Count'].shift(1)
df_final = merged.dropna()

print(f"✅ Success! Training set contains {len(df_final)} trading days.")

Fetching news and prices starting from 2026-02-20...
Processed News Page 1...
API Limit reached or error: {'status': 'error', 'code': 'maximumResultsReached', 'message': 'You have requested too many results. Developer accounts are limited to a max of 100 results. You are trying to request results 100 to 200. Please upgrade to a paid plan if you need more results.'}


[*********************100%***********************]  1 of 1 completed

✅ Success! Training set contains 18 trading days.


In [51]:
# %% [Cell 4: Build & Train TensorFlow Model]
def build_model():
    model = models.Sequential([
        layers.Input(shape=(2,)), # Inputs: Sentiment Score & Article Count
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(8, activation='relu'),
        layers.Dense(1, activation='linear') # Linear for regression (predicting % return)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Prepare Data
X = df_final[['Prev_Sentiment', 'Prev_Count']].values
y = df_final['Return'].values

# Normalize Count
X[:, 1] = X[:, 1] / (X[:, 1].max() + 1)

# Initialize and Train
forecaster = build_model()
print("\nTraining Model...")
history = forecaster.fit(X, y, epochs=50, batch_size=2, verbose=0)
print("Training Complete.")


Training Model...
Training Complete.


In [52]:
# %% [Cell 5: Live Prediction for Tomorrow]
# Fetch the most recent news batch to see "today's mood"
today_news = newsapi.get_everything(q='Amazon AND stock', language='en', sort_by='publishedAt', page_size=20)
today_scores = [get_sentiment(a['title']) for a in today_news['articles']]

if today_scores:
    current_avg = np.mean(today_scores)
    current_count = len(today_scores) / (df_final['Article_Count'].max() + 1)

    pred_input = np.array([[current_avg, current_count]])
    prediction = forecaster.predict(pred_input, verbose=0)[0][0]

    print(f"\n--- PREDICTION FOR NEXT TRADING SESSION ---")
    print(f"Current Sentiment: {current_avg:.4f}")
    print(f"Predicted Return:  {prediction * 100:.4f}%")
    print(f"Action: {'BUY/LONG 🟢' if prediction > 0 else 'SELL/SHORT 🔴'}")
else:
    print("Not enough news found for a live prediction.")


--- PREDICTION FOR NEXT TRADING SESSION ---
Current Sentiment: 0.0859
Predicted Return:  1.6915%
Action: BUY/LONG 🟢
